### Set Up

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, ttest_ind
from functools import reduce
import os
import sys

In [2]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

In [29]:
scenarios_path = Path().resolve() / "../../scenarios_inputs" / "cheung_variants" 
annotated_output_path = Path().resolve() / "../../annotated_outputs" / "cheung_variants"
plots_path = Path().resolve() / "../../analysis" / "cheung_variants" / "cheung_visualizations"
print(f"scenarios_path: {scenarios_path}")
print(f"annotated_output_path: {annotated_output_path}")



scenarios_path: /Users/emstrictly/Library/CloudStorage/Dropbox/2025_moral_scenario_annotation/code/em/graph_extract/analysis/cheung_variants/../../scenarios_inputs/cheung_variants
annotated_output_path: /Users/emstrictly/Library/CloudStorage/Dropbox/2025_moral_scenario_annotation/code/em/graph_extract/analysis/cheung_variants/../../annotated_outputs/cheung_variants


In [4]:
ROOT_DIR = os.getcwd() + '/../../'
sys.path.append(ROOT_DIR)
sys.path.append(ROOT_DIR+'src/')


In [28]:
ROOT_DIR

'/Users/emstrictly/Library/CloudStorage/Dropbox/2025_moral_scenario_annotation/code/em/graph_extract/analysis/cheung_variants/../../'

In [6]:
import analysis_utils
import src.generic_analysis_utils as generic_analysis_utils

### Define Functions

In [7]:
def read_scenario(file_path): 

    narrative, id = generic_analysis_utils.parse_filename_cheung(file_path)


    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    nodes = [json.loads(line.strip()) for line in lines if line.strip()]

    act_choice = generic_analysis_utils.extract_action(nodes)


    # also return original scenario text for reference
    with open(scenarios_path / f"{narrative}.json") as f:

        scenario_list = json.load(f)
        #get the one with the matching id
        this_scenario = next(s for s in scenario_list if s['id'] == id)
     

        #extract the condition from the json read in
        scenario_text = this_scenario['text']
        scenario_deontology = this_scenario['deontology_level']
        scenario_utility = this_scenario['utility_level']
        
    #create a little dictionary with everything in it
    scenario_info = {
        "narrative": narrative,
        "id": id,
        "act": act_choice,
        "text": scenario_text,
        "deontology": scenario_deontology,
        "utility": scenario_utility
    }

    return nodes, scenario_info

In [8]:
def single_scenario_analysis(scenario_name, this_id, act_id):

    filename = filename_template % (scenario_name, this_id, act_id)

    file_path = annotated_output_path / filename


    nodes, scenario_dictionary = read_scenario(file_path)
    
    print(f"Read scenario: {file_path}")

    print(f"Scenario number: {scenario_dictionary['id']}")
    print(f"Scenario action choice: {scenario_dictionary['act']}")
    print(f"Scenario utility: {scenario_dictionary['utility']}")
    print(f"Scenario deontology: {scenario_dictionary['deontology']}")
    util_df =  generic_analysis_utils.events_to_utility_df(nodes)


    #get the column means for the utility dataframe (per entity)
    print(util_df.mean())

    util_mean = np.mean(util_df.mean())
    print('overall mean: ' + str(util_mean))

    return util_mean

In [9]:
def get_counterfact_util(scenario_id):

    util_mean_1 = single_scenario_analysis(SCENARIO_NAME, scenario_id,1)
    print('\n\n')
    util_mean_2 = single_scenario_analysis(SCENARIO_NAME, scenario_id,2)

    util_diff = util_mean_1 -  util_mean_2
    print('\n\nutility difference between choice 1 and choice 2: ' + str(util_diff))

    return util_diff


In [10]:
def create_agg_df():

    rows = []

    for scenario_id in SCENARIO_LIST:


            #read in scenario action choice 1
            this_filename = filename_template % (SCENARIO_NAME, scenario_id, 1)
            file_path = annotated_output_path / this_filename
            nodes, scenario_dictionary = read_scenario(file_path)


            # #get condition labels
            # print(f"Scenario number: {scenario_dictionary['id']}")
            # print(f"Scenario action choice: {scenario_dictionary['act']}")
            # print(f"Scenario utility: {scenario_dictionary['utility']}")
            # print(f"Scenario deontology: {scenario_dictionary['deontology']}")


            #get utility from this action
            util_df =  generic_analysis_utils.events_to_utility_df(nodes)
            util_mean_1 = np.mean(util_df.mean())

            #read in scenario action choice 2
            this_filename = filename_template % (SCENARIO_NAME, scenario_id, 2)
            file_path = annotated_output_path / this_filename
            nodes_2, scenario_dictionary_2 = read_scenario(file_path)
            #get utility from this action
            util_df =  generic_analysis_utils.events_to_utility_df(nodes_2)
            util_mean_2 = np.mean(util_df.mean())


            util_diff = util_mean_1 -  util_mean_2



            rows.append(
                {
                    "scenario_id": scenario_dictionary.get("id"),
                    "narrative": scenario_dictionary.get("narrative"),
                    "deontology_label": scenario_dictionary.get("deontology"),
                    "utility_label": scenario_dictionary.get("utility"),
                    "deontology_rating": generic_analysis_utils.extract_deontology(nodes),
                    "utility_rating": util_diff,        
                }
            )

    results_df = pd.DataFrame(rows)
    results_df.sort_values(by="scenario_id", inplace=True)
    # Ensure numeric columns are numeric
    results_df["deontology_rating"] = pd.to_numeric(results_df["deontology_rating"], errors="coerce")
    results_df["utility_rating"] = pd.to_numeric(results_df["utility_rating"], errors="coerce")

    return results_df

### Select scenario to analyze

In [13]:
#get all of the json files in the annotated_output_path
SCENARIO_NAME = "lifeboat"
json_files = list(annotated_output_path.glob(f"*{SCENARIO_NAME}_*choice_1.json"))
SCENARIO_LIST = list(range(1, int(len(json_files)) + 1))
#note, we assume that there are 2 action ids, where id 1 is the action and id 2 is the counterfactual action. 
filename_template = "%s_%d_choice_%d.json"

In [34]:
os.getcwd()
annotated_output_path
bel=os.path.join(annotated_output_path,'bird_1_choice_1.json')
bel

'/Users/emstrictly/Library/CloudStorage/Dropbox/2025_moral_scenario_annotation/code/em/graph_extract/analysis/cheung_variants/../../annotated_outputs/cheung_variants/bird_1_choice_1.json'

In [41]:
generic_analysis_utils.read_annotation(bel)[6]

{'node': {'kind': 'event', 'label': 'I experience intense sadness'},
 'links': [{'link': {'kind': 'utility', 'value': '-70'}, 'to_node': 'i'},
  {'link': {'kind': 'utility', 'value': '0'}, 'to_node': 'the bird'}]}

### Extract utility and deontology scores from individual scanerios to examine them closely

In [12]:
get_counterfact_util(1)

Read scenario: /Users/emstrictly/Library/CloudStorage/Dropbox/2025_moral_scenario_annotation/code/em/graph_extract/analysis/cheung_variants/../../annotated_outputs/cheung_variants/lifeboat_1_choice_1.json
Scenario number: 1
Scenario action choice: physically throw 10 passengers from lifeboat B overboard
Scenario utility: 3
Scenario deontology: 1
i                                                            -1.428571
the other occupants of lifeboat a                           -18.571429
10 passengers in lifeboat b who would be thrown overboard   -63.571429
51 remaining occupants of lifeboat b                          1.857143
dtype: float64
overall mean: -20.428571428571427



Read scenario: /Users/emstrictly/Library/CloudStorage/Dropbox/2025_moral_scenario_annotation/code/em/graph_extract/analysis/cheung_variants/../../annotated_outputs/cheung_variants/lifeboat_1_choice_2.json
Scenario number: 1
Scenario action choice: do nothing
Scenario utility: 3
Scenario deontology: 1
i, the captain

np.float64(0.508928571428573)

In [ ]:
get_counterfact_util(4)

In [ ]:
get_counterfact_util(7)

In [ ]:
get_counterfact_util(2)

In [ ]:
get_counterfact_util(5)

In [ ]:
get_counterfact_util(8)

In [ ]:
get_counterfact_util(9)

In [ ]:
get_counterfact_util(6)

In [ ]:
get_counterfact_util(3)

### Get aggregate utilities and arrange into dataframe

In [ ]:
results_df = create_agg_df()
results_df.sort_values(by="deontology_label", inplace=True)
results_df

In [ ]:
# 1) Means/SEs by deontology_label (for both dependent measures)
summary_by_deontology = (
    results_df
    .groupby("deontology_label", dropna=False)
    .agg(
        n=("deontology_rating", "count"),
        deontology_rating_mean=("deontology_rating", "mean"),
        deontology_rating_se=("deontology_rating", "sem"),
        utility_rating_mean=("utility_rating", "mean"),
        utility_rating_se=("utility_rating", "sem"),
    )
    .reset_index()
    .sort_values("deontology_label")
)

# 2) Means/SEs by utility_label (for both dependent measures)
summary_by_utility = (
    results_df
    .groupby("utility_label", dropna=False)
    .agg(
        n=("deontology_rating", "count"),
        deontology_rating_mean=("deontology_rating", "mean"),
        deontology_rating_se=("deontology_rating", "sem"),
        utility_rating_mean=("utility_rating", "mean"),
        utility_rating_se=("utility_rating", "sem"),
    )
    .reset_index()
    .sort_values("utility_label")
)


In [ ]:
summary_by_deontology

In [ ]:
summary_by_utility

In [ ]:
## get plot

# Long format for plotting both dependent measures together
plot_df = results_df.melt(
    id_vars=["deontology_label", "utility_label"],
    value_vars=["deontology_rating", "utility_rating"],
    var_name="measure",
    value_name="value",
)


# Ensure label columns are ordered numerically for plotting
deontology_order = sorted(plot_df["deontology_label"].dropna().astype(int).unique())
utility_order = sorted(plot_df["utility_label"].dropna().astype(int).unique())

plot_df["deontology_label"] = pd.Categorical(
    plot_df["deontology_label"].astype(int),
    categories=deontology_order,
    ordered=True
)
plot_df["utility_label"] = pd.Categorical(
    plot_df["utility_label"].astype(int),
    categories=utility_order,
    ordered=True
)
analysis_utils.make_deont_util_plot(SCENARIO_NAME, plot_df, plots_path)